In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.spatial.distance import cdist
from scipy.stats import norm
import time
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

In [ ]:
X, y = load_wine(return_X_y=True)

In [10]:
print(f"Dataset shape: {X.shape}")
print(f"Classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")

Dataset shape: (178, 13)
Classes: [0 1 2]
Class distribution: [59 71 48]


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train, y_train)
y_pred_baseline = rf_baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

In [ ]:
print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred_baseline)}")

Baseline accuracy: 1.0000

Confusion Matrix:
[[12  0  0]
 [ 0 14  0]
 [ 0  0 10]]


In [ ]:
#GRID SEARCH
class GridSearch:
    """Grid search over hyperparameter space."""
    
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.results = []
        self.best_score = 0
        self.best_params = None
    
    def evaluate(self, n_estimators, max_depth, min_samples_split):
        rf = RandomForestClassifier(
            n_estimators=int(n_estimators),
            max_depth=int(max_depth) if max_depth < 100 else None,
            min_samples_split=int(min_samples_split),
            random_state=42
        )
        rf.fit(self.X_train, self.y_train)
        y_pred = rf.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        return accuracy
    
    def search(self):
        print("Running Grid Search...")
        
        n_estimators_list = [10, 50, 100, 200, 500]
        max_depth_list = [3, 5, 10, 15, 20, 30]
        min_samples_split_list = [2, 5, 10, 20]
        
        iteration = 0
        for n_est in n_estimators_list:
            for max_d in max_depth_list:
                for min_samp in min_samples_split_list:
                    score = self.evaluate(n_est, max_d, min_samp)
                    self.results.append({
                        'iteration': iteration,
                        'n_estimators': n_est,
                        'max_depth': max_d,
                        'min_samples_split': min_samp,
                        'accuracy': score
                    })
                    
                    if score > self.best_score:
                        self.best_score = score
                        self.best_params = {
                            'n_estimators': n_est,
                            'max_depth': max_d,
                            'min_samples_split': min_samp
                        }
                    
                    iteration += 1
                    if iteration % 20 == 0:
                        print(f"  Iteration {iteration}, Best: {self.best_score:.4f}")
        
        print(f" Grid Search complete. Total evals: {iteration}")
        print(f"  Best accuracy: {self.best_score:.4f}")
        print(f"  Best params: {self.best_params}")
        
        return pd.DataFrame(self.results)
 

gs = GridSearch(X_train, X_test, y_train, y_test)
start_time = time.time()
gs_results = gs.search()
gs_time = time.time() - start_time
 
print(f"Execution time: {gs_time:.2f}s")
print(f"\nTop 5 results:")
print(gs_results.nlargest(5, 'accuracy'))

Running Grid Search...
  Iteration 20, Best: 1.0000
  Iteration 40, Best: 1.0000
  Iteration 60, Best: 1.0000
  Iteration 80, Best: 1.0000
  Iteration 100, Best: 1.0000
  Iteration 120, Best: 1.0000
✓ Grid Search complete. Total evals: 120
  Best accuracy: 1.0000
  Best params: {'n_estimators': 10, 'max_depth': 3, 'min_samples_split': 2}
Execution time: 78.52s

Top 5 results:
   iteration  n_estimators  max_depth  min_samples_split  accuracy
0          0            10          3                  2       1.0
1          1            10          3                  5       1.0
2          2            10          3                 10       1.0
3          3            10          3                 20       1.0
4          4            10          5                  2       1.0


In [ ]:
class RandomSearch:
    """Random search for hyperparameter tuning."""
    
    def __init__(self, X_train, X_test, y_train, y_test, n_iterations=50):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.n_iterations = n_iterations
        self.results = []
        self.best_score = 0
        self.best_params = None
    
    def evaluate(self, n_estimators, max_depth, min_samples_split):
        rf = RandomForestClassifier(
            n_estimators=int(n_estimators),
            max_depth=int(max_depth) if max_depth < 100 else None,
            min_samples_split=int(min_samples_split),
            random_state=42
        )
        rf.fit(self.X_train, self.y_train)
        y_pred = rf.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        return accuracy
    
    def search(self):
        print(f"Running Random Search ({self.n_iterations} iterations)...")
        
        for iteration in range(self.n_iterations):
            n_est = np.random.randint(10, 501)
            max_d = np.random.randint(2, 31)
            min_samp = np.random.randint(2, 21)
            
            score = self.evaluate(n_est, max_d, min_samp)
            self.results.append({
                'iteration': iteration,
                'n_estimators': n_est,
                'max_depth': max_d,
                'min_samples_split': min_samp,
                'accuracy': score
            })
            
            if score > self.best_score:
                self.best_score = score
                self.best_params = {
                    'n_estimators': n_est,
                    'max_depth': max_d,
                    'min_samples_split': min_samp
                }
            
            if (iteration + 1) % 10 == 0:
                print(f"  Iteration {iteration + 1}, Best: {self.best_score:.4f}")
        
        print(f" Random Search complete. Total evals: {self.n_iterations}")
        print(f"  Best accuracy: {self.best_score:.4f}")
        print(f"  Best params: {self.best_params}")
        
        return pd.DataFrame(self.results)
 
# Run it
rs = RandomSearch(X_train, X_test, y_train, y_test, n_iterations=50)
start_time = time.time()
rs_results = rs.search()
rs_time = time.time() - start_time
 
print(f"Execution time: {rs_time:.2f}s")
print(f"\nTop 5 results:")
print(rs_results.nlargest(5, 'accuracy'))

Running Random Search (50 iterations)...
  Iteration 10, Best: 1.0000
  Iteration 20, Best: 1.0000
  Iteration 30, Best: 1.0000
  Iteration 40, Best: 1.0000
  Iteration 50, Best: 1.0000
✓ Random Search complete. Total evals: 50
  Best accuracy: 1.0000
  Best params: {'n_estimators': 112, 'max_depth': 21, 'min_samples_split': 16}
Execution time: 50.24s

Top 5 results:
   iteration  n_estimators  max_depth  min_samples_split  accuracy
0          0           112         21                 16       1.0
1          1           116          9                  8       1.0
2          2           131         20                 12       1.0
3          3           468         25                  5       1.0
4          4           369         25                  4       1.0
